In [2]:
%load_ext autoreload
%autoreload 2
import sys, warnings
sys.path.append('..')
warnings.filterwarnings('ignore')

import pandas as pd
import pm4py
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from python_analysis.functions import (
    read_log, distinct_values, show_trace_rows, unique_analysis, field_summary, traces_without,
    trace_share, start_end, event_code_legend, variants,
    activity_table, equivalence_classes, template, cover_table, assign_classes,
    metric, discover_and_evaluate, compare, combine_partitions, style_stats,
    discover_inductive_noise_twentieth, discover_ilp, discover_powl, discover_inductive_noise_tenth,
    process_durations, bottleneck_analysis, rejection_analysis, unexpected_behavior,
    conformance_analysis, compare_declarations,
)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [3]:
##Daten lesen
dom_log_base = read_log('../data/DomesticDeclarations.xes')
int_log_base = read_log('../data/InternationalDeclarations.xes')

dummy = int_log_base["case:Permit id"].ne(int_log_base["case:Permit ID"])
decl_and_permit = int_log_base[~dummy].copy()
international_only_decl = int_log_base[dummy].copy()
declarations_only = pd.concat(
    [dom_log_base.assign(**{"case:source": "domestic"}),
     international_only_decl.assign(**{"case:source": "international"})],
    ignore_index=True,
).sort_values(["case:concept:name", "time:timestamp"], kind="stable", ignore_index=True)

logs = {"domestic": dom_log_base,
        "domestic_without_D20": dom_log_base[~dom_log_base["concept:name"].eq("Declaration SAVED by EMPLOYEE")],
        "international": int_log_base,
        "decl_only": declarations_only,
        "decl_only_without_trips": declarations_only[~declarations_only["concept:name"].isin({"Start trip", "End trip"})],
        "decl_and_permit": decl_and_permit,
        "decl_and_permit_without_trips": decl_and_permit[~decl_and_permit["concept:name"].isin({"Start trip", "End trip"})],
        "decl_and_permit_without_trips_or_D20": decl_and_permit[~decl_and_permit["concept:name"].isin({"Start trip", "End trip", "Declaration SAVED by EMPLOYEE"})],
        "international_only_decl": international_only_decl,
        "international_only_decl_without_trips": international_only_decl[~international_only_decl["concept:name"].isin({"Start trip", "End trip"})],
        "international_only_decl_without_trips_or_D20": international_only_decl[~international_only_decl["concept:name"].isin({"Start trip", "End trip", "Declaration SAVED by EMPLOYEE"})],
        }

parsing log, completed traces :: 100%|██████████| 6449/6449 [00:01<00:00, 3580.33it/s]


In [4]:
##Datenfelder je Trace / Event
field_summary(logs).style \
    .set_caption("Datenfelder je Trace / Event") \
    .set_properties(**{"text-align": "left"})

,Log,Eventfelder,Tracefelder
0,domestic,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount']"
1,domestic_without_D20,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount']"
2,international,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
3,decl_only,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'case:source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
4,decl_only_without_trips,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'case:source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
5,decl_and_permit,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
6,decl_and_permit_without_trips,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
7,decl_and_permit_without_trips_or_D20,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
8,international_only_decl,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:co

In [5]:
print("\nDomestic Log (10500):")
for f in dom_log_base.columns:
    print(str(f) + ' : '+ str(len(distinct_values(dom_log_base, f))))

print("\nInternational Log (6449):")
for f in int_log_base.columns:
    print(str(f) + ' : '+ str(len(distinct_values(int_log_base, f))))


Domestic Log (10500):
id : 56437
org:resource : 2
concept:name : 17
time:timestamp : 45403
org:role : 7
case:id : 10500
case:concept:name : 10500
case:BudgetNumber : 1
case:DeclarationNumber : 10049
case:Amount : 8326

International Log (6449):
id : 69073
org:resource : 2
concept:name : 34
time:timestamp : 51270
org:role : 8
case:Permit travel permit number : 5596
case:DeclarationNumber : 6190
case:Amount : 6100
case:RequestedAmount : 6100
case:Permit TaskNumber : 6
case:Permit BudgetNumber : 207
case:OriginalAmount : 6100
case:Permit ProjectNumber : 825
case:concept:name : 6449
case:Permit OrganizationalEntity : 27
case:travel permit number : 6033
case:Permit RequestedBudget : 5259
case:id : 6449
case:Permit ID : 6028
case:Permit id : 5608
case:BudgetNumber : 719
case:Permit ActivityNumber : 145
case:AdjustedAmount : 6101


In [6]:
unique_analysis(int_log_base, ["case:OriginalAmount", "case:AdjustedAmount", "case:RequestedAmount", "case:Amount"])

,case:OriginalAmount,case:AdjustedAmount,case:RequestedAmount,case:Amount,traces
0,0.000000,0.000000,0.000000,0.000000,2251
1,45.654385,45.654385,45.654385,45.654385,73
2,64.473928,64.473928,64.473928,64.473928,72
3,49.536371,49.536371,49.536371,49.536371,58
4,35.122357,35.122357,35.122357,35.122357,52


True     6448
False       1
Name: count, dtype: int64


,case:id,case:OriginalAmount,case:AdjustedAmount,case:RequestedAmount,case:Amount
0,declaration 54434,0.0,100.494292,0.0,0.0


['declaration 54434']

In [7]:
#Permit Dummy Analyse 448 Traces
problematic = set(unique_analysis(int_log_base, ["case:Permit id", "case:Permit ID"]))
without_permit = traces_without(int_log_base, "Permit")

print("Traces ohne Permit-Event:", len(without_permit))
print("problematic_traces:", len(problematic))
print("Gleiche Menge:", without_permit == problematic)
print("Nur ohne Permit:", sorted(without_permit - problematic))
print("Nur in problematic_traces:", sorted(problematic - without_permit))

,case:Permit id,case:Permit ID,traces
0,travel permit 73562,travel permit 73562,170
1,travel permit 53343,travel permit 53343,140
2,travel permit 32577,travel permit 32577,115
3,travel permit 45618,travel permit 45618,98
4,travel permit 64232,travel permit 64232,84


True     6001
False     448
Name: count, dtype: int64


,case:id,case:Permit id,case:Permit ID
0,declaration 143637,travel permit 423,travel permit 143640
1,declaration 143644,travel permit 423,travel permit 143647
2,declaration 143578,travel permit 423,travel permit 143581
3,declaration 143585,travel permit 423,travel permit 143588
4,declaration 143657,travel permit 423,travel permit 143660
...,...,...,...
443,declaration 147452,travel permit 423,travel permit 147455
444,declaration 147470,travel permit 423,travel permit 147473
445,declaration 147487,travel permit 423,travel permit 147490
446,declaration 147507,travel permit 423,travel permit 147509


Traces ohne Permit-Event: 448
problematic_traces: 448
Gleiche Menge: True
Nur ohne Permit: []
Nur in problematic_traces: []


In [8]:
# Erst alle Aktivitäten anzeigen
event_types = pd.concat([pd.Series(dom_log_base["concept:name"].values), pd.Series(int_log_base["concept:name"].values)]).drop_duplicates().reset_index(drop=True).sort_values(ignore_index=True)
display(event_types)

# Dann Relationen finden: Wörter mit identischer Aktivitätsmenge bilden eine Äquivalenzklasse
acts = activity_table({"domestic": dom_log_base, "international": int_log_base})
eq = equivalence_classes(acts)
display(eq)

# Masken/Templates je Klasse definieren
eq["template"] = eq["members"].apply(template)
display(eq)

# Unnötige Masken entfernen, weil sie alles/nichts matchen
filtered_groups = eq[~eq["template"].str.fullmatch(r"[\s%]*", na=True)]
display(filtered_groups)

# Gruppen an Masken definieren, die genau jede Aktivität genau einmal abdecken
covers = cover_table(filtered_groups, set(acts.index))
display(covers)

acts["entity_classes"] = assign_classes(acts, filtered_groups, covers.at[1, "ids"])
acts["activity_classes"] = assign_classes(acts, filtered_groups, covers.at[11, "ids"])

0         Declaration APPROVED by ADMINISTRATION
1           Declaration APPROVED by BUDGET OWNER
2           Declaration APPROVED by PRE_APPROVER
3             Declaration APPROVED by SUPERVISOR
4         Declaration FINAL_APPROVED by DIRECTOR
5       Declaration FINAL_APPROVED by SUPERVISOR
6     Declaration FOR_APPROVAL by ADMINISTRATION
7       Declaration FOR_APPROVAL by PRE_APPROVER
8         Declaration FOR_APPROVAL by SUPERVISOR
9         Declaration REJECTED by ADMINISTRATION
10          Declaration REJECTED by BUDGET OWNER
11              Declaration REJECTED by DIRECTOR
12              Declaration REJECTED by EMPLOYEE
13               Declaration REJECTED by MISSING
14          Declaration REJECTED by PRE_APPROVER
15            Declaration REJECTED by SUPERVISOR
16                 Declaration SAVED by EMPLOYEE
17             Declaration SUBMITTED by EMPLOYEE
18                                      End trip
19                               Payment Handled
20             Permi

,words,n,domestic,international,members
0,MEMBER STAFF,34,36353,59347,"[Declaration APPROVED by ADMINISTRATION, Decla..."
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla..."
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla..."
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP..."
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla..."
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla..."
6,EMPLOYEE,7,13031,29338,"[Declaration REJECTED by EMPLOYEE, Declaration..."
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati..."
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla..."
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara..."


,words,n,domestic,international,members,template
0,MEMBER STAFF,34,36353,59347,"[Declaration APPROVED by ADMINISTRATION, Decla...",%
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by %
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla...",Declaration % by %
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP...",Permit % by %
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla...",% REJECTED by %
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla...",% APPROVED by %
6,EMPLOYEE,7,13031,29338,"[Declaration REJECTED by EMPLOYEE, Declaration...",%
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati...",% by SUPERVISOR
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by ADMINISTRATION
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara...",% by PRE_APPROVER


,words,n,domestic,international,members,template
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by %
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla...",Declaration % by %
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP...",Permit % by %
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla...",% REJECTED by %
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla...",% APPROVED by %
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati...",% by SUPERVISOR
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by ADMINISTRATION
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara...",% by PRE_APPROVER
10,BUDGET OWNER,4,2879,3668,"[Declaration APPROVED by BUDGET OWNER, Declara...",% by BUDGET OWNER
11,DIRECTOR,4,0,897,"[Declaration FINAL_APPROVED by DIRECTOR, Decla...",% by DIRECTOR


,n_groups,groups,ids
0,4,"[BY, PAYMENT, TRIP, REMINDER SEND]","[1, 16, 18, 21]"
1,5,"[DECLARATION, PERMIT, PAYMENT, TRIP, REMINDER ...","[2, 3, 16, 18, 21]"
2,5,"[BY, TRIP, HANDLED, REMINDER SEND, REQUEST]","[1, 18, 20, 21, 22]"
3,5,"[BY, PAYMENT, END, REMINDER SEND, START]","[1, 16, 19, 21, 24]"
4,6,"[DECLARATION, PERMIT, TRIP, HANDLED, REMINDER ...","[2, 3, 18, 20, 21, 22]"
5,6,"[DECLARATION, PERMIT, PAYMENT, END, REMINDER S...","[2, 3, 16, 19, 21, 24]"
6,6,"[BY, END, HANDLED, REMINDER SEND, REQUEST, START]","[1, 19, 20, 21, 22, 24]"
7,7,"[DECLARATION, PERMIT, END, HANDLED, REMINDER S...","[2, 3, 19, 20, 21, 22, 24]"
8,9,"[REJECTED, APPROVED, FINAL_APPROVED, FOR_APPRO...","[4, 5, 12, 13, 16, 17, 18, 21, 23]"
9,10,"[REJECTED, APPROVED, FINAL_APPROVED, FOR_APPRO...","[4, 5, 12, 13, 17, 18, 20, 21, 22, 23]"


Menschliche Bewertung:
| index | Ausschlusskriterium? | Bewertung |
|-------|----------------------|-----------|
| 0     |Ja, durch "By" -> Permit/Declaration fälschlich zusammengefasst||
| 1     || Sehr gut, genau 5 Gruppen für 5 "menschlich erkennbare" [Entitätsklassen]|
| 2     |Ja, durch "By", s. (1)||
| 3     |Ja, durch "By", s. (1)||
| 4     |Ja, Aktivitäten und Entitäten werden gemischt "End, Payment, Start, etc."||
| 5     ||Ist okay, im Prinzip Gruppe (1), aber mit extra Aufteilung von "%Payment%-Aktivitäten"|
| 6     |Ja, durch "By", s. (1)||
| 7     |s. (4)||
| 8     |Ja, ähnlich zu (4) werden Aktivitäten und Entitäten gemischt (Trip, Reminder, Payment + Submitted, Approval...)||
| 9     |Ja, ebenfalls noch Entitäten (Trip) und Aktivitäten||
| 10    ||Fast gültig, es verbleiben noch eine zwei, die eineindeutige Aktivität (Reminder Send) und Payment, sonst alles abgedeckt|
| 11    ||Teilt wie bei (1) zu (5) nur das Payment auf, wodurch aber nur noch eine Entität mit eindeutiger Aktivität verbleibt -> Perfekt als [Aktivitätenklassen]|


Es verbleibt noch ein Problem, was nicht direkt ersichtlich ist: Durch die vorhandenen Matcher entsteht bei der Betrachtung der Events in ihren Klassen ein Informationsverlust bei den Aktivitäten durch den Matcher "%APPROVED %", deswegen darf beim Aufbau der Klassen nicht einfach eine Transformation zu Buchstaben erfolgen, sondern der Unterstrich beibehalten werden.

Die entstehenden Gruppierungen zeigen außerdem direkt auf, dass eindeutige Unterscheidungen nach Akteuren nicht zu einer allgemeingültigen bijektiven Abbildung führen und demnach nicht **eigenständig** geeignet für isolierte Analysen sind.

In [9]:
acts.groupby(["entity_classes", "activity_classes"])[["domestic", "international"]].sum()

domestic  international
entity_classes     activity_classes                                      
% Payment %        Payment Handled                   10044           6187
                   Request Payment                   10040           6183
% trip             End trip                              0           6449
                   Start trip                            0           6449
Declaration % by % % APPROVED by %                   11707           7739
                   % FINAL_APPROVED by %             10131           6291
                   % REJECTED by %                    2846           3686
                   % SUBMITTED by EMPLOYEE           11531           8099
                   Declaration FOR_APPROVAL by %         3              0
                   Declaration SAVED by EMPLOYEE       135             75
Permit % by %      % APPROVED by %                       0           7777
                   % FINAL_APPROVED by %                 0           6021
                   % REJECTED by %                       0            506
                   % SUBMITTED by EMPLOYEE               0           6255
Send Reminder      Send Reminder                         0            434

In [10]:
trace_share(logs, "concept:name")

,domestic,domestic_without_D20,international,decl_only,decl_only_without_trips,decl_and_permit,decl_and_permit_without_trips,decl_and_permit_without_trips_or_D20,international_only_decl,international_only_decl_without_trips,international_only_decl_without_trips_or_D20
concept:name,,,,,,,,,,,
Declaration APPROVED by ADMINISTRATION,7983/10500 [76.03%],7983/10366 [77.01%],4919/6449 [76.28%],8234/10948 [75.21%],8234/10948 [75.21%],4668/6001 [77.79%],4668/6001 [77.79%],4668/6001 [77.79%],251/448 [56.03%],251/448 [56.03%],251/438 [57.31%]
Declaration APPROVED by BUDGET OWNER,2807/10500 [26.73%],2807/10366 [27.08%],1817/6449 [28.17%],2924/10948 [26.71%],2924/10948 [26.71%],1700/6001 [28.33%],1700/6001 [28.33%],1700/6001 [28.33%],117/448 [26.12%],117/448 [26.12%],117/438 [26.71%]
Declaration APPROVED by PRE_APPROVER,666/10500 [6.34%],666/10366 [6.42%],558/6449 [8.65%],712/10948 [6.50%],712/10948 [6.50%],512/6001 [8.53%],512/6001 [8.53%],512/6001 [8.53%],46/448 [10.27%],46/448 [10.27%],46/438 [10.50%]
Declaration APPROVED by SUPERVISOR,,,246/6449 [3.81%],34/10948 [0.31%],34/10948 [0.31%],212/6001 [3.53%],212/6001 [3.53%],212/6001 [3.53%],34/448 [7.59%],34/448 [7.59%],34/438 [7.76%]
Declaration FINAL_APPROVED by DIRECTOR,,,244/6449 [3.78%],34/10948 [0.31%],34/10948 [0.31%],210/6001 [3.50%],210/6001 [3.50%],210/6001 [3.50%],34/448 [7.59%],34/448 [7.59%],34/438 [7.76%]
Declaration FINAL_APPROVED by SUPERVISOR,10070/10500 [95.90%],10070/10366 [97.14%],5961/6449 [92.43%],10463/10948 [95.57%],10463/10948 [95.57%],5568/6001 [92.78%],5568/6001 [92.78%],5568/6001 [92.78%],393/448 [87.72%],393/448 [87.72%],393/438 [89.73%]
Declaration FOR_APPROVAL by ADMINISTRATION,1/10500 [0.01%],1/10366 [0.01%],,1/10948 [0.01%],1/10948 [0.01%],,,,,,
Declaration FOR_APPROVAL by PRE_APPROVER,1/10500 [0.01%],1/10366 [0.01%],,1/10948 [0.01%],1/10948 [0.01%],,,,,,
Declaration FOR_APPROVAL by SUPERVISOR,1/10500 [0.01%],1/10366 [0.01%],,1/10948 [0.01%],1/10948 [0.01%],,,,,,


In [11]:
display(event_code_legend(logs))

with pd.option_context("display.max_colwidth", None):
    for name, log in logs.items():
        table = variants(log)
        print("" + "=" * 70)
        print(f"\n{name}: {len(table)} Abläufe")
        display(table)

,concept:name,Code
0,Declaration APPROVED by ADMINISTRATION,D36
1,Declaration APPROVED by BUDGET OWNER,D31
2,Declaration APPROVED by PRE_APPROVER,D32
3,Declaration APPROVED by SUPERVISOR,D35
4,Declaration FINAL_APPROVED by DIRECTOR,D5☑7
5,Declaration FINAL_APPROVED by SUPERVISOR,D5☑5
6,Declaration FOR_APPROVAL by ADMINISTRATION,D46
7,Declaration FOR_APPROVAL by PRE_APPROVER,D42
8,Declaration FOR_APPROVAL by SUPERVISOR,D45
9,Declaration REJECTED by ADMINISTRATION,D5❌6



domestic: 99 Abläufe


,Ablauf,Traces,Anteil
0,D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,4618,4618/10500 [43.98%]
1,D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,2473,2473/10500 [23.55%]
2,D10 > D5☑5 > 🪙❓ > 🪙❗,1392,1392/10500 [13.26%]
3,D10 > D32 > D5☑5 > 🪙❓ > 🪙❗,575,575/10500 [5.48%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,345,345/10500 [3.29%]
...,...,...,...
94,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5 > D5❌_,1,1/10500 [0.01%]
95,D10 > D5☑5 > D5❌_ > D10 > D5❌5 > D5❌0 > D10 > D5☑5 > 🪙❓ > 🪙❗,1,1/10500 [0.01%]
96,D20 > 🪙❓ > 🪙❗,1,1/10500 [0.01%]
97,D10 > D42 > D5❌_,1,1/10500 [0.01%]



domestic_without_D20: 98 Abläufe


,Ablauf,Traces,Anteil
0,D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,4618,4618/10366 [44.55%]
1,D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,2473,2473/10366 [23.86%]
2,D10 > D5☑5 > 🪙❓ > 🪙❗,1392,1392/10366 [13.43%]
3,D10 > D32 > D5☑5 > 🪙❓ > 🪙❗,575,575/10366 [5.55%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,345,345/10366 [3.33%]
...,...,...,...
93,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5 > D5❌_,1,1/10366 [0.01%]
94,D10 > D5☑5 > D5❌_ > D10 > D5❌5 > D5❌0 > D10 > D5☑5 > 🪙❓ > 🪙❗,1,1/10366 [0.01%]
95,🪙❓ > 🪙❗,1,1/10366 [0.01%]
96,D10 > D42 > D5❌_,1,1/10366 [0.01%]


...

,Ablauf,Traces,Anteil
0,D10 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,111,111/448 [24.78%]
1,D10 > D36 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,79,79/448 [17.63%]
2,D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,75,75/448 [16.74%]
3,D10 > D32 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,33,33/448 [7.37%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,21,21/448 [4.69%]
5,D10 > D35 > D5☑7 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,14,14/448 [3.12%]
6,T⬆️ > T⬇️ > D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,12,12/448 [2.68%]
7,D10 > D36 > D35 > D5☑7 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,11,11/448 [2.46%]
8,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗ > T⬆️ > T⬇️,9,9/448 [2.01%]
9,D20 > T⬆️ > T⬇️,8,8/448 [1.79%]



international_only_decl_without_trips: 33 Abläufe


,Ablauf,Traces,Anteil
0,D10 > D5☑5 > 🪙❓ > 🪙❗,111,111/448 [24.78%]
1,D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,90,90/448 [20.09%]
2,D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,88,88/448 [19.64%]
3,D10 > D32 > D5☑5 > 🪙❓ > 🪙❗,33,33/448 [7.37%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,29,29/448 [6.47%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,16,16/448 [3.57%]
6,D10 > D35 > D5☑7 > 🪙❓ > 🪙❗,14,14/448 [3.12%]
7,D10 > D36 > D35 > D5☑7 > 🪙❓ > 🪙❗,11,11/448 [2.46%]
8,D20,10,10/448 [2.23%]
9,D10 > D5❌6 > D5❌0,10,10/448 [2.23%]



international_only_decl_without_trips_or_D20: 32 Abläufe


,Ablauf,Traces,Anteil
0,D10 > D5☑5 > 🪙❓ > 🪙❗,111,111/438 [25.34%]
1,D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,90,90/438 [20.55%]
2,D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,88,88/438 [20.09%]
3,D10 > D32 > D5☑5 > 🪙❓ > 🪙❗,33,33/438 [7.53%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5 > 🪙❓ > 🪙❗,29,29/438 [6.62%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5 > 🪙❓ > 🪙❗,16,16/438 [3.65%]
6,D10 > D35 > D5☑7 > 🪙❓ > 🪙❗,14,14/438 [3.20%]
7,D10 > D36 > D35 > D5☑7 > 🪙❓ > 🪙❗,11,11/438 [2.51%]
8,D10 > D5❌6 > D5❌0,10,10/438 [2.28%]
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5 > 🪙❓ > 🪙❗,6,6/438 [1.37%]


In [12]:
start_end(logs)

In [13]:
MINERS = {
    "inductivepn": pm4py.discover_petri_net_inductive,
    "alphapn": pm4py.discover_petri_net_alpha,
    "alphapluspn": pm4py.discover_petri_net_alpha_plus,
    "heuristicpn": pm4py.discover_petri_net_heuristics,
    "inductivebpmn": pm4py.discover_bpmn_inductive,
    "inductive_noise10pn": discover_inductive_noise_tenth,
    "inductive_noise20pn": discover_inductive_noise_twentieth,
    "ilppn": discover_ilp,
    "powl": discover_powl,
}

models, stats = {}, {}
for name, log in logs.items():
    for miner, discover in MINERS.items():
        models[(name, miner)], stats[(name, miner)] = discover_and_evaluate(name, miner, discover, log)

table = pd.DataFrame(stats)
table.columns.names = ["Log", "Verfahren"]
display(style_stats(compare(table)))
display(style_stats(table))

domestic_inductivepn_model.pkl: aus Cache geladen
domestic_inductivepn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_alphapn_model.pkl: aus Cache geladen
domestic_alphapn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_alphapluspn_model.pkl: aus Cache geladen
domestic_alphapluspn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_heuristicpn_model.pkl: aus Cache geladen
domestic_heuristicpn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductivebpmn_model.pkl: aus Cache geladen
domestic_inductivebpmn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductive_noise10pn_model.pkl: aus Cache geladen
domestic_inductive_noise10pn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductive_noise20pn_model.pkl: aus Cache geladen
domestic_inductive_noise20pn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_ilppn_model.pkl: aus Cache geladen
domestic_ilppn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_powl_model.pkl: aus Cache geladen
domestic_powl_stats.pkl: 19 aus Cache

In [14]:
# Log-Gruppen, die jeweils zusammen alle Traces abdecken: raw vs. vorverarbeitet vs. ohne Trips vs. drei Prozesse getrennt
PARTITIONS = {
    "raw": ["domestic", "international"],
    "vorverarbeitet": ["decl_only", "decl_and_permit"],
    "ohne Trips": ["decl_only_without_trips", "decl_and_permit_without_trips"],
    "getrennt ohne Trips": ["domestic", "decl_and_permit_without_trips", "international_only_decl_without_trips"],
    "getrennt ohne Trips, ohne SAVE": ["domestic_without_D20", "decl_and_permit_without_trips_or_D20", "international_only_decl_without_trips_or_D20"],
}
combined = combine_partitions(table, logs, PARTITIONS)
display(style_stats(combined[["heuristicpn"]]))
display(style_stats(combined))


### Trace-Varianten je Log: nur Declaration

Alle anderen Events werden vor der Variantenbildung entfernt. Die Ablaeufe verwenden die Kurzcodes aus der Legende oben; die Anteile beziehen sich auf Traces mit Declaration-Events.

In [15]:
declaration_variants = {}
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    for name, log in logs.items():
        declaration_events = log[log["concept:name"].str.contains(r"\bDeclaration\b", na=False)]
        declaration_variants[name] = variants(declaration_events)
        print("=" * 70)
        print(f"\n{name}: {len(declaration_variants[name])} Declaration-Ablaeufe")
        display(declaration_variants[name])


domestic: 94 Declaration-Ablaeufe


,Ablauf,Traces,Anteil
0,D10 > D36 > D5☑5,4622,4622/10500 [44.02%]
1,D10 > D36 > D31 > D5☑5,2475,2475/10500 [23.57%]
2,D10 > D5☑5,1392,1392/10500 [13.26%]
3,D10 > D32 > D5☑5,575,575/10500 [5.48%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5,345,345/10500 [3.29%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5,188,188/10500 [1.79%]
6,D10 > D5❌6 > D5❌0,174,174/10500 [1.66%]
7,D20,135,135/10500 [1.29%]
8,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D5☑5,77,77/10500 [0.73%]
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5,57,57/10500 [0.54%]



domestic_without_D20: 93 Declaration-Ablaeufe


,Ablauf,Traces,Anteil
0,D10 > D36 > D5☑5,4622,4622/10365 [44.59%]
1,D10 > D36 > D31 > D5☑5,2475,2475/10365 [23.88%]
2,D10 > D5☑5,1392,1392/10365 [13.43%]
3,D10 > D32 > D5☑5,575,575/10365 [5.55%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5,345,345/10365 [3.33%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5,188,188/10365 [1.81%]
6,D10 > D5❌6 > D5❌0,174,174/10365 [1.68%]
7,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D5☑5,77,77/10365 [0.74%]
8,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5,57,57/10365 [0.55%]
9,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D31 > D5☑5,49,49/10365 [0.47%]



international: 111 Declaration-Ablaeufe


...

,Ablauf,Traces,Anteil
0,D10 > D5☑5,111,111/448 [24.78%]
1,D10 > D36 > D5☑5,90,90/448 [20.09%]
2,D10 > D36 > D31 > D5☑5,90,90/448 [20.09%]
3,D10 > D32 > D5☑5,33,33/448 [7.37%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5,29,29/448 [6.47%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5,16,16/448 [3.57%]
6,D10 > D35 > D5☑7,14,14/448 [3.12%]
7,D10 > D36 > D35 > D5☑7,11,11/448 [2.46%]
8,D20,10,10/448 [2.23%]
9,D10 > D5❌6 > D5❌0,10,10/448 [2.23%]



international_only_decl_without_trips: 32 Declaration-Ablaeufe


,Ablauf,Traces,Anteil
0,D10 > D5☑5,111,111/448 [24.78%]
1,D10 > D36 > D5☑5,90,90/448 [20.09%]
2,D10 > D36 > D31 > D5☑5,90,90/448 [20.09%]
3,D10 > D32 > D5☑5,33,33/448 [7.37%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5,29,29/448 [6.47%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5,16,16/448 [3.57%]
6,D10 > D35 > D5☑7,14,14/448 [3.12%]
7,D10 > D36 > D35 > D5☑7,11,11/448 [2.46%]
8,D20,10,10/448 [2.23%]
9,D10 > D5❌6 > D5❌0,10,10/448 [2.23%]



international_only_decl_without_trips_or_D20: 31 Declaration-Ablaeufe


,Ablauf,Traces,Anteil
0,D10 > D5☑5,111,111/438 [25.34%]
1,D10 > D36 > D5☑5,90,90/438 [20.55%]
2,D10 > D36 > D31 > D5☑5,90,90/438 [20.55%]
3,D10 > D32 > D5☑5,33,33/438 [7.53%]
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5☑5,29,29/438 [6.62%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5☑5,16,16/438 [3.65%]
6,D10 > D35 > D5☑7,14,14/438 [3.20%]
7,D10 > D36 > D35 > D5☑7,11,11/438 [2.51%]
8,D10 > D5❌6 > D5❌0,10,10/438 [2.28%]
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5☑5,6,6/438 [1.37%]


In [16]:
import re
from collections import Counter
from python_analysis.functions import variants

domestic_without_trips = logs["domestic"][
    ~logs["domestic"]["concept:name"].isin(["Start trip", "End trip"])
]
declaration_events = domestic_without_trips[
    domestic_without_trips["concept:name"].str.contains(
        r"\bDeclaration\b", na=False
    )
]

def split_variants(log, event: str):
    counts = Counter()
    for row in variants(log).itertuples(index=False):
        # D10 bleibt erhalten; ein D10 am Trace-Anfang erzeugt keinen leeren Teil.
        parts = re.split(r" > (?="+event+"(?: > |$))", row.Ablauf)
        for part in parts:
            counts[part] += row.Traces

    split_variants = (
        pd.DataFrame(counts.items(), columns=["Ablauf", "Teil-Traces"])
        .sort_values("Teil-Traces", ascending=False, ignore_index=True)
    )
    total = split_variants["Teil-Traces"].sum()
    split_variants["Anteil"] = split_variants["Teil-Traces"].map(
        lambda n: f"{n}/{total} [{n / total:.2%}]"
    )

    return split_variants

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df = split_variants(declaration_events, "D10") # Declaration Submitted by EMPLOYEE
    display(df)

,Ablauf,Teil-Traces,Anteil
0,D10 > D36 > D5☑5,5133,5133/11666 [44.00%]
1,D10 > D36 > D31 > D5☑5,2788,2788/11666 [23.90%]
2,D10 > D5☑5,1462,1462/11666 [12.53%]
3,D10 > D5❌6 > D5❌0,928,928/11666 [7.95%]
4,D10 > D32 > D5☑5,659,659/11666 [5.65%]
5,D10 > D36 > D5❌5 > D5❌0,185,185/11666 [1.59%]
6,D20,135,135/11666 [1.16%]
7,D10 > D5❌2 > D5❌0,85,85/11666 [0.73%]
8,D10 > D5☑5 > D5❌_,74,74/11666 [0.63%]
9,D10 > D5❌5 > D5❌0,60,60/11666 [0.51%]


In [17]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df = variants(declaration_events)
    display(df.loc[df["Ablauf"].str.contains("D20", regex=False, na=False)])

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df = split_variants(declaration_events, "D20") # Declaration SAVED by EMPLOYEE
    #display(df)
    display(df.loc[df["Ablauf"].str.contains("D20", na=False)])

,Ablauf,Traces,Anteil
7,D20,135,135/10500 [1.29%]


,Ablauf,Teil-Traces,Anteil
7,D20,135,135/10500 [1.29%]


Aufgabe 2

In [26]:
# Frage 2: volle Ausgangslogs, unabhaengig von Miner und Discovery-Aufteilung.
# Zeiten = erster bis letzter beobachteter Event, nicht zwingend abgeschlossene Faelle.
# Zellenangaben unten zaehlen alle Notebook-Zellen ab 1, nicht Ausfuehrungsnummern.
analysis_logs = {"domestic": dom_log_base, "international": int_log_base}
duration_results = {}
for name, log in analysis_logs.items():
    duration_results[name] = process_durations(log)
    print(name)
    display(duration_results[name]["summary"])
    #display(duration_results[name]["by_start_month"]) #Randmonate eventuell nicht repräsentant
    #display(duration_results[name]["cases"].head(20))

domestic


,cases,mean_days,median_days,min_days,max_days,p90_days
group,,,,,,
all,10500.0,11.525722,7.330289,0.0,469.236319,20.120839


international


,cases,mean_days,median_days,min_days,max_days,p90_days
group,,,,,,
all,6449.0,86.451729,66.041667,6.729988,742.0,159.185472


Aufgabe 3

In [19]:
# Frage 3: direkt aufeinanderfolgende Events mit langen Zeitabstaenden.
# Keine Trennung von Warte-, Arbeits- und Reisezeit, keine automatische Ursachenbestimmung.
# Alle Paare bleiben im Ergebnis; angezeigt werden je 20 nach Median und Gesamtzeit.
bottleneck_results = {}
for name, log in analysis_logs.items():
    bottleneck_results[name] = bottleneck_analysis(log)
    print(name, '- Engpass-Kandidaten, Zeiten in Tagen')
    transitions = bottleneck_results[name]["transitions"]
    display(transitions.head(20))
    display(transitions.sort_values("total_days", ascending=False).head(20))
    display(bottleneck_results[name]["intervals"].head(20))

domestic - Engpass-Kandidaten, Zeiten in Tagen


,,occurrences,cases,mean_days,median_days,p90_days,max_days,total_days,case_share
from_activity,to_activity,,,,,,,,
Declaration FINAL_APPROVED by SUPERVISOR,Payment Handled,7,7,81.585701,48.366100,147.397583,284.274653,571.099907,0.000667
Request Payment,Declaration REJECTED by MISSING,3,3,18.986154,26.765324,29.304361,29.939120,56.958461,0.000286
Declaration SAVED by EMPLOYEE,Request Payment,1,1,17.033345,17.033345,17.033345,17.033345,17.033345,0.000095
Declaration REJECTED by SUPERVISOR,Request Payment,1,1,6.922326,6.922326,6.922326,6.922326,6.922326,0.000095
Declaration REJECTED by BUDGET OWNER,Declaration SUBMITTED by EMPLOYEE,1,1,6.092315,6.092315,6.092315,6.092315,6.092315,0.000095
Declaration FOR_APPROVAL by PRE_APPROVER,Declaration REJECTED by MISSING,1,1,4.132477,4.132477,4.132477,4.132477,4.132477,0.000095
Declaration APPROVED by BUDGET OWNER,Declaration REJECTED by SUPERVISOR,32,32,4.189010,3.996985,6.836884,6.849745,134.048322,0.003048
Request Payment,Payment Handled,10037,10037,3.574593,3.240475,4.912648,62.145127,35878.189363,0.955905
Declaration FINAL_APPROVED by SUPERVISOR,Declaration REJECTED by MISSING,86,82,11.029341,3.060185,13.878374,293.360428,948.523356,0.007810


,,occurrences,cases,mean_days,median_days,p90_days,max_days,total_days,case_share
from_activity,to_activity,,,,,,,,
Request Payment,Payment Handled,10037,10037,3.574593,3.240475,4.912648,62.145127,35878.189363,0.955905
Declaration FINAL_APPROVED by SUPERVISOR,Request Payment,10038,10038,3.162912,1.142836,6.974372,234.680012,31749.310370,0.956000
Declaration APPROVED by ADMINISTRATION,Declaration FINAL_APPROVED by SUPERVISOR,5133,5133,1.902123,0.869850,4.767502,49.923634,9763.595405,0.488857
Declaration SUBMITTED by EMPLOYEE,Declaration APPROVED by ADMINISTRATION,8202,7983,1.155718,0.000428,0.016400,282.582558,9479.195347,0.760286
Declaration APPROVED by BUDGET OWNER,Declaration FINAL_APPROVED by SUPERVISOR,2788,2788,2.864106,1.909751,6.720473,11.852593,7985.128530,0.265524
Declaration REJECTED by EMPLOYEE,Declaration SUBMITTED by EMPLOYEE,1081,961,5.178999,0.202211,11.066655,356.922269,5598.498032,0.091524
Declaration SUBMITTED by EMPLOYEE,Declaration REJECTED by ADMINISTRATION,952,846,5.662438,0.001105,1.694793,469.236319,5390.641192,0.080571
Declaration APPROVED by ADMINISTRATION,Declaration APPROVED by BUDGET OWNER,2820,2807,1.897848,0.865469,4.859507,49.929745,5351.930787,0.267333
Declaration REJECTED by ADMINISTRATION,Declaration REJECTED by EMPLOYEE,928,837,2.043734,0.922176,5.040957,27.162870,1896.584826,0.079714


,case:concept:name,from_activity,start,to_activity,end,elapsed_days
720,declaration 100723,Declaration SUBMITTED by EMPLOYEE,2018-01-08 15:24:37+00:00,Declaration REJECTED by ADMINISTRATION,2019-04-22 21:04:55+00:00,469.236319
1977,declaration 101989,Declaration REJECTED by EMPLOYEE,2017-06-20 09:22:05+00:00,Declaration SUBMITTED by EMPLOYEE,2018-06-12 07:30:09+00:00,356.922269
199,declaration 100197,Declaration FINAL_APPROVED by SUPERVISOR,2017-03-21 04:15:02+00:00,Declaration REJECTED by MISSING,2018-01-08 12:54:03+00:00,293.360428
15554,declaration 115669,Declaration FINAL_APPROVED by SUPERVISOR,2018-04-26 09:55:46+00:00,Payment Handled,2019-02-04 16:31:16+00:00,284.274653
26056,declaration 126246,Declaration SUBMITTED by EMPLOYEE,2018-07-01 17:47:14+00:00,Declaration APPROVED by ADMINISTRATION,2019-04-10 07:46:07+00:00,282.582558
54599,declaration 98152,Declaration SUBMITTED by EMPLOYEE,2017-12-04 16:59:49+00:00,Declaration REJECTED by ADMINISTRATION,2018-08-27 17:16:11+00:00,266.011366
2466,declaration 102487,Declaration SUBMITTED by EMPLOYEE,2018-03-09 08:34:28+00:00,Declaration REJECTED by ADMINISTRATION,2018-11-07 18:19:12+00:00,243.406065
55738,declaration 99294,Declaration SUBMITTED by EMPLOYEE,2018-03-09 08:52:30+00:00,Declaration REJECTED by ADMINISTRATION,2018-11-07 18:19:12+00:00,243.393542
52166,declaration 95691,Declaration FINAL_APPROVED by SUPERVISOR,2017-05-15 17:56:22+00:00,Request Payment,2018-01-05 10:15:35+00:00,234.680012
3587,declaration 103613,Declaration SUBMITTED by EMPLOYEE,2018-04-11 13:25:39+00:00,Declaration REJECTED by ADMINISTRATION,2018-11-12 13:27:20+00:00,215.001169


international - Engpass-Kandidaten, Zeiten in Tagen


occurrences  \
from_activity                            to_activity                                         
Declaration SAVED by EMPLOYEE            End trip                                        2   
Declaration FINAL_APPROVED by SUPERVISOR Payment Handled                                 5   
End trip                                 Permit APPROVED by PRE_APPROVER                 2   
Payment Handled                          End trip                                      121   
                                         Start trip                                    415   
End trip                                 Permit REJECTED by MISSING                      5   
Declaration REJECTED by MISSING          Start trip                                      3   
Declaration SAVED by EMPLOYEE            Start trip                                     16   
Declaration REJECTED by MISSING          End trip                                        1   
Permit FINAL_APPROVED by DIRECTOR        Start trip                                    493   
End trip                                 Send Reminder                                 288   
Permit FINAL_APPROVED by DIRECTOR        Send Reminder                                   3   
Declaration REJECTED by EMPLOYEE         Start trip                                     36   
Permit REJECTED by MISSING               Start trip                                      2   
Permit FINAL_APPROVED by SUPERVISOR      Declaration APPROVED by PRE_APPROVER            1   
Declaration REJECTED by EMPLOYEE         End trip                                       10   
Declaration REJECTED by MISSING          Permit REJECTED by MISSING                      2   
Permit FINAL_APPROVED by SUPERVISOR      Send Reminder                                 101   
                                         Start trip                                   4206   
Permit FINAL_APPROVED by DIRECTOR        Declaration SUBMITTED by EMPLOYEE              95   

                                                                               cases  \
from_activity                            to_activity                                   
Declaration SAVED by EMPLOYEE            End trip                                  2   
Declaration FINAL_APPROVED by SUPERVISOR Payment Handled                           5   
End trip                                 Permit APPROVED by PRE_APPROVER           2   
Payment Handled                          End trip                                121   
                                         Start trip                              415   
End trip                                 Permit REJECTED by MISSING                5   
Declaration REJECTED by MISSING          Start trip                                3   
Declaration SAVED by EMPLOYEE            Start trip                               16   
Declaration REJECTED by MISSING          End trip                                  1   
Permit FINAL_APPROVED by DIRECTOR        Start trip                              493   
End trip                                 Send Reminder                           288   
Permit FINAL_APPROVED by DIRECTOR        Send Reminder                             3   
Declaration REJECTED by EMPLOYEE         Start trip                               36   
Permit REJECTED by MISSING               Start trip                                2   
Permit FINAL_APPROVED by SUPERVISOR      Declaration APPROVED by PRE_APPROVER      1   
Declaration REJECTED by EMPLOYEE         End trip                                 10   
Declaration REJECTED by MISSING          Permit REJECTED by MISSING                2   
Permit FINAL_APPROVED by SUPERVISOR      Send Reminder                           101   
                                         Start trip                             4206   
Permit FINAL_APPROVED by DIRECTOR        Declaration SUBMITTED by EMPLOYEE        95   

                                                                         

,,occurrences,cases,mean_days,median_days,p90_days,max_days,total_days,case_share
from_activity,to_activity,,,,,,,,
Permit FINAL_APPROVED by SUPERVISOR,Start trip,4206,4206,39.251807,27.412442,90.518148,338.575521,165093.101204,0.652194
Payment Handled,Start trip,415,415,182.145684,77.228275,417.227836,452.228113,75590.458715,0.064351
End trip,Declaration SUBMITTED by EMPLOYEE,4591,4591,12.698465,5.648299,28.514456,295.891898,58298.653692,0.711893
Start trip,End trip,5784,5784,5.222402,4.000000,9.000000,274.000000,30206.375000,0.896883
Permit FINAL_APPROVED by DIRECTOR,Start trip,493,493,54.812778,43.387627,115.655826,272.371389,27022.699745,0.076446
Request Payment,Payment Handled,6149,6149,3.456923,3.217812,4.917162,31.116354,21256.619039,0.953481
Declaration FINAL_APPROVED by SUPERVISOR,Request Payment,5918,5918,3.207570,1.187025,7.476610,266.099039,18982.401412,0.917662
Payment Handled,End trip,121,121,123.865694,82.228229,276.311678,660.269410,14987.748970,0.018763
End trip,Permit SUBMITTED by EMPLOYEE,492,492,26.301193,9.363414,61.746005,366.472373,12940.186794,0.076291


,case:concept:name,from_activity,start,to_activity,end,elapsed_days
47684,declaration 61184,Payment Handled,2018-07-19 15:32:03+00:00,End trip,2020-05-09 22:00:00+00:00,660.269410
3630,declaration 143612,Declaration SAVED by EMPLOYEE,2017-01-10 13:16:45+00:00,Start trip,2018-04-09 22:00:00+00:00,454.363368
3589,declaration 143578,Payment Handled,2017-01-12 16:31:31+00:00,Start trip,2018-04-09 22:00:00+00:00,452.228113
3621,declaration 143598,Payment Handled,2017-01-16 16:32:18+00:00,Start trip,2018-04-09 22:00:00+00:00,448.227569
3654,declaration 143637,Payment Handled,2017-01-16 16:32:18+00:00,Start trip,2018-04-09 22:00:00+00:00,448.227569
3636,declaration 143617,Payment Handled,2017-01-19 16:31:50+00:00,Start trip,2018-04-09 22:00:00+00:00,445.227894
3684,declaration 143670,Payment Handled,2017-01-19 16:31:50+00:00,Start trip,2018-04-09 22:00:00+00:00,445.227894
3642,declaration 143624,Payment Handled,2017-01-23 16:32:17+00:00,Start trip,2018-04-09 22:00:00+00:00,441.227581
3481,declaration 143467,Payment Handled,2017-01-23 16:32:17+00:00,Start trip,2018-04-09 22:00:00+00:00,441.227581
3497,declaration 143474,Payment Handled,2017-01-23 16:32:17+00:00,Start trip,2018-04-09 22:00:00+00:00,441.227581


Aufgabe 4

In [20]:
# Frage 4: Declaration-Ablehnungen; Permit-Entscheidungen werden nicht mitgezaehlt.
# Bezugsgroesse: Traces mit Declaration-Events, nicht Zahl der REJECTED-Events.
# Ein REJECTED-Event ist keine endgueltige Ablehnung. Spaetere Freigaben separat.
# Bei gleichen Zeitstempeln gilt fuer die Entscheidungsfolge die stabile Logreihenfolge.
# Ablehnende Aktivitaet/Rolle und Budgetgruppen liefern Hinweise, keine Ursachen.
rejection_results = {}
for name, log in analysis_logs.items():
    rejection_results[name] = rejection_analysis(log, group_by="case:BudgetNumber")
    print(name)
    display(rejection_results[name]["summary"])
    display(rejection_results[name]["by_activity"])
    display(rejection_results[name]["by_group"].head(20))
    cases = rejection_results[name]["cases"]
    display(cases.loc[cases["rejected"]].head(20))
# Zelle 3 (field_summary) zeigt kein explizites Ablehnungsgrund-Feld.
# Auch last_entity_event_rejected bedeutet nur letzter beobachteter Declaration-Status.

domestic


,cases,rejection_events,rejected_cases,rejected_share,final_approval_after_last_rejection,last_entity_event_rejected
entity,,,,,,
Declaration,10500,2846,1301,0.123905,978,323


,rejection_events,rejected_cases,case_share
concept:name,,,
Declaration REJECTED by ADMINISTRATION,952,846,0.080571
Declaration REJECTED by BUDGET OWNER,59,58,0.005524
Declaration REJECTED by EMPLOYEE,1365,1212,0.115429
Declaration REJECTED by MISSING,91,87,0.008286
Declaration REJECTED by PRE_APPROVER,86,81,0.007714
Declaration REJECTED by SUPERVISOR,293,281,0.026762


,cases,rejected_cases,rejected_share
case:BudgetNumber,,,
budget 86566,10500,1301,0.123905


,rejection_events,last_entity_activity,rejected,final_approval_after_last_rejection,last_entity_event_rejected,case:BudgetNumber
case:concept:name,,,,,,
declaration 100122,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100197,1,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100245,4,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100365,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100380,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100399,4,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 86566
declaration 100433,4,Declaration REJECTED by EMPLOYEE,True,False,True,budget 86566
declaration 100512,2,Declaration REJECTED by EMPLOYEE,True,False,True,budget 86566
declaration 100543,2,Declaration REJECTED by EMPLOYEE,True,False,True,budget 86566


international


,cases,rejection_events,rejected_cases,rejected_share,final_approval_after_last_rejection,last_entity_event_rejected
entity,,,,,,
Declaration,6449,3686,1576,0.244379,1389,186


,rejection_events,rejected_cases,case_share
concept:name,,,
Declaration REJECTED by ADMINISTRATION,1549,1287,0.199566
Declaration REJECTED by BUDGET OWNER,40,40,0.006203
Declaration REJECTED by DIRECTOR,4,4,0.000620
Declaration REJECTED by EMPLOYEE,1780,1483,0.229958
Declaration REJECTED by MISSING,103,98,0.015196
Declaration REJECTED by PRE_APPROVER,84,82,0.012715
Declaration REJECTED by SUPERVISOR,126,122,0.018918


,cases,rejected_cases,rejected_share
case:BudgetNumber,,,
budget 146697,58,15,0.258621
budget 146776,56,14,0.250000
budget 146681,54,14,0.259259
budget 146666,53,18,0.339623
budget 145339,49,2,0.040816
budget 146757,42,12,0.285714
budget 146663,42,16,0.380952
budget 146690,42,18,0.428571
budget 146682,38,13,0.342105


,rejection_events,last_entity_activity,rejected,final_approval_after_last_rejection,last_entity_event_rejected,case:BudgetNumber
case:concept:name,,,,,,
declaration 10089,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146762
declaration 10168,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146817
declaration 10240,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 147421
declaration 10265,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 147063
declaration 10279,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146755
declaration 10324,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 147423
declaration 10495,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146850
declaration 10522,4,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146668
declaration 10552,2,Declaration FINAL_APPROVED by SUPERVISOR,True,True,False,budget 146681


Aufgabe 5

In [21]:
# Frage 5: bereits analysiert in Zelle 6 (#Permit Dummy Analyse):
# 448 internationale Traces ohne Permit-Event; exakt dieselben 448 haben
# abweichende case:Permit id / case:Permit ID. Kein Beweis fehlender realer Genehmigung.
# Zelle 5 (unique_analysis der Betraege): nur declaration 54434 weicht ab;
# AdjustedAmount = 100.494292, die drei anderen verglichenen Betragsfelder = 0.
# Ergaenzung: explizite Pruefhypothese, fachlich noch zu bestaetigen:
# Vor dem ersten Start trip muss eine finale Permit-Freigabe beobachtet worden sein.
permit_rules = {
    "Finale Permit-Freigabe vor Reisebeginn": (
        {"Permit FINAL_APPROVED by DIRECTOR", "Permit FINAL_APPROVED by SUPERVISOR"},
        {"Start trip"},
    ),
}
unexpected_results = unexpected_behavior(int_log_base, rules=permit_rules)
display(unexpected_results["summary"])
display(unexpected_results["cases"].query("status not in ['ok', 'no_trigger']").head(30))
# Gleiche Zeitstempel sind unklar; Faelle ohne Start trip sind nicht bewertbar.
# Nur das vollstaendige internationale Log verwenden, keine Events herausfiltern.

,cases,eligible_cases,violations,violation_share,missing_required,required_after_trigger,same_timestamp,no_trigger
Finale Permit-Freigabe vor Reisebeginn,6449,6449,1578,0.244689,448,1130,0,0


,case:concept:name,required_time,trigger_time,status,rule
4,declaration 10069,2018-02-22 07:27:05+00:00,2018-02-21 23:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
10,declaration 10151,2018-02-26 14:09:04+00:00,2018-02-18 23:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
11,declaration 10168,2018-09-06 07:51:39+00:00,2018-08-25 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
18,declaration 10279,2018-09-18 03:15:13+00:00,2018-08-19 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
29,declaration 10472,2018-06-19 08:56:26+00:00,2018-06-17 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
33,declaration 10552,2018-07-17 03:15:12+00:00,2018-07-07 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
35,declaration 10578,2018-06-18 08:58:33+00:00,2018-06-10 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
40,declaration 10638,2018-06-26 05:26:44+00:00,2018-06-12 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
50,declaration 10743,2018-09-25 07:09:50+00:00,2018-09-13 22:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn
54,declaration 1079,2018-03-13 04:15:17+00:00,2018-03-07 23:00:00+00:00,required_after_trigger,Finale Permit-Freigabe vor Reisebeginn


Aufgabe 6

In [22]:
# Frage 6: Zellen 13/14 enthalten bereits globale Modellkennzahlen.
# Hier zusaetzlich konkrete Fall-Diagnosen und haeufige nicht passende Varianten.
# Beispielauswahl, keine automatische Bestmodell-Auswahl; in Zelle 13 vorhanden.
conformance_miner = "heuristicpn"
conformance_results = {}
for name, log in analysis_logs.items():
    reference_model = models[(name, conformance_miner)]
    conformance_results[name] = conformance_analysis(log, model=reference_model)
    print(name, '- Referenzmodell:', conformance_miner)
    display(conformance_results[name]["summary"])
    patterns = conformance_results[name]["patterns"]
    display(patterns.loc[patterns["non_fitting_cases"].gt(0)].head(20))
    cases = conformance_results[name]["cases"]
    display(cases.loc[~cases["trace_is_fit"]].head(20))
# Modelle wurden aus denselben Logs entdeckt: keine unabhaengige Compliance-Pruefung.
# Ein fachliches Sollmodell kann stattdessen direkt als model uebergeben werden.
# Token-Fitness allein kann unbekannte Aktivitaeten uebersehen; diese stehen separat im Ergebnis.

domestic - Referenzmodell: heuristicpn


,cases,fitting_cases,non_fitting_cases,fitting_share,mean_fitness,cases_with_unknown_activities
0,10500,3714,6786,0.353714,0.918933,0


,cases,fitting_cases,mean_fitness,missing_tokens,remaining_tokens,non_fitting_cases,non_fitting_share
variant,,,,,,,
"(Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",4618,0,0.875000,4618,4618,4618,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",1392,0,0.857143,1392,1392,1392,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by ADMINISTRATION, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",345,0,0.923077,345,345,345,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration REJECTED by SUPERVISOR, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",77,0,0.866667,154,154,77,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration REJECTED by SUPERVISOR, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration APPROVED by BUDGET OWNER, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",49,0,0.941176,49,49,49,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Declaration REJECTED by MISSING, Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",40,0,0.833333,80,80,40,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by ADMINISTRATION, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by ADMINISTRATION, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",36,0,0.944444,36,36,36,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration REJECTED by SUPERVISOR, Declaration REJECTED by EMPLOYEE)",31,0,0.888889,31,31,31,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by SUPERVISOR, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",22,0,0.833333,44,44,22,1.0


,trace_is_fit,trace_fitness,missing_tokens,remaining_tokens,consumed_tokens,produced_tokens,unknown_activities,variant
case:concept:name,,,,,,,,
declaration 100000,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100005,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100010,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100027,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100032,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100037,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100048,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100059,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."
declaration 100064,False,0.875,1,1,8,8,(),"(Declaration SUBMITTED by EMPLOYEE, Declaratio..."


international - Referenzmodell: heuristicpn


,cases,fitting_cases,non_fitting_cases,fitting_share,mean_fitness,cases_with_unknown_activities
0,6449,354,6095,0.054892,0.896192,0


,cases,fitting_cases,mean_fitness,missing_tokens,remaining_tokens,non_fitting_cases,non_fitting_share
variant,,,,,,,
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",1369,0,0.878571,2738,4107,1369,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit APPROVED by BUDGET OWNER, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration APPROVED by BUDGET OWNER, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",624,0,0.979167,0,624,624,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",361,0,0.882353,722,722,361,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by ADMINISTRATION, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",311,0,0.898333,622,933,311,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit APPROVED by BUDGET OWNER, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration REJECTED by ADMINISTRATION, Declaration REJECTED by EMPLOYEE, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration APPROVED by BUDGET OWNER, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",188,0,0.982143,0,188,188,1.0
"(Start trip, End trip, Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit FINAL_APPROVED by SUPERVISOR, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",157,0,0.791667,471,785,157,1.0
"(Declaration SUBMITTED by EMPLOYEE, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled, Start trip, End trip)",111,0,0.833333,222,222,111,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit FINAL_APPROVED by SUPERVISOR, Start trip, End trip, Send Reminder, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",94,0,0.826087,376,376,94,1.0
"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED by ADMINISTRATION, Permit APPROVED by SUPERVISOR, Permit FINAL_APPROVED by DIRECTOR, Start trip, End trip, Declaration SUBMITTED by EMPLOYEE, Declaration APPROVED by ADMINISTRATION, Declaration FINAL_APPROVED by SUPERVISOR, Request Payment, Payment Handled)",85,0,0.878571,170,255,85,1.0


,trace_is_fit,trace_fitness,missing_tokens,remaining_tokens,consumed_tokens,produced_tokens,unknown_activities,variant
case:concept:name,,,,,,,,
declaration 1002,False,0.878571,2,3,20,21,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10024,False,0.979167,0,1,23,24,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10026,False,0.979167,0,1,23,24,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10046,False,0.878571,2,3,20,21,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10069,False,0.743464,4,5,17,18,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10079,False,0.878571,2,3,20,21,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10089,False,0.851449,3,4,23,24,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10106,False,0.923077,2,2,26,26,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."
declaration 10128,False,0.878571,2,3,20,21,(),"(Permit SUBMITTED by EMPLOYEE, Permit APPROVED..."


Aufgabe 7

In [23]:
# Frage 7: bereits vorhanden: Zelle 10 (Aktivitaetsanteile), 11 (Varianten),
# 12 (Start/Ende) und 18 (Declaration-Varianten). Zelle 11 zeigt in den gespeicherten
# Ergebnissen 99 domestic- gegen 753 international-Ablaeufe im vollen Log.
# Das allein ist kein fairer Komplexitaetsvergleich: international enthaelt auch Permits/Trips.
# Ergaenzung: Zeit-, Ablehnungs- und Strukturkennzahlen fuer beide Ausgangslogs;
# Declaration-Spannen und volle Trace-Dauern werden getrennt ausgewiesen.
declaration_comparison = compare_declarations(dom_log_base, int_log_base)
for label, result in declaration_comparison.items():
    print(label)
    display(result)
# Anteile sind Zahlen zwischen 0 und 1; Differenzen * 100 ergeben Prozentpunkte.
# Declaration-Spannen enden am letzten Declaration-Event, nicht zwingend an der Zahlung.
# Deskriptive Unterschiede, kein Signifikanz- oder Kausalitaetsnachweis.

overview


,domestic,international
all_cases,10500.000000,6449.000000
declaration_cases,10500.000000,6449.000000
cases_without_declaration,0.000000,0.000000
full_median_days,7.330289,66.041667
full_p90_days,20.120839,159.185472
declaration_median_days,1.545972,3.645949
declaration_p90_days,9.617816,17.795139
declaration_mean_events,3.462190,4.014576
declaration_variants,94.000000,111.000000
most_common_variant_share,0.440190,0.358970


declaration_activity_shares


,domestic,international,international_minus_domestic
concept:name,,,
Declaration APPROVED by ADMINISTRATION,0.760286,0.762754,0.002468
Declaration APPROVED by BUDGET OWNER,0.267333,0.281749,0.014416
Declaration APPROVED by PRE_APPROVER,0.063429,0.086525,0.023096
Declaration APPROVED by SUPERVISOR,0.000000,0.038145,0.038145
Declaration FINAL_APPROVED by DIRECTOR,0.000000,0.037835,0.037835
Declaration FINAL_APPROVED by SUPERVISOR,0.959048,0.924329,-0.034718
Declaration FOR_APPROVAL by ADMINISTRATION,0.000095,0.000000,-0.000095
Declaration FOR_APPROVAL by PRE_APPROVER,0.000095,0.000000,-0.000095
Declaration FOR_APPROVAL by SUPERVISOR,0.000095,0.000000,-0.000095


declaration_by_start_month


cases  mean_days  median_days  min_days  \
log           start_month                                            
domestic      2017-01        122   3.450613     0.004161  0.000000   
              2017-02        168   4.263271     0.000949  0.000023   
              2017-03        191   1.971969     0.013021  0.000000   
              2017-04        174   1.437844     0.001921  0.000000   
              2017-05        201   4.850752     0.002766  0.000000   
              2017-06        225   4.993548     0.009340  0.000000   
              2017-07        149   4.117909     0.004051  0.000000   
              2017-08         87   2.326216     0.001400  0.000023   
              2017-09        222   2.377300     0.000712  0.000000   
              2017-10        215   2.361564     0.008924  0.000000   
              2017-11        244   2.615174     0.001574  0.000000   
              2017-12        242   5.078754     0.005116  0.000000   
              2018-01        564   6.396998     1.970590  0.000000   
              2018-02        669   5.937706     1.978843  0.000000   
              2018-03        685   5.136924     1.792558  0.000000   
              2018-04        756   5.177850     1.974844  0.000000   
              2018-05        634   5.605149     1.944468  0.000000   
              2018-06        914   5.851919     2.726377  0.000000   
              2018-07        682   6.223602     1.955208  0.000000   
              2018-08        296   6.066717     2.749479  0.000000   
              2018-09        641   5.396917     2.855949  0.000000   
              2018-10        804   4.631662     2.578652  0.000000   
              2018-11        851   4.861393     2.841817  0.000000   
              2018-12        764   5.182176     2.499971  0.000000   
international 2017-01         53   3.399232     0.024699  0.000000   
              2017-02        120   1.046497     0.003275  0.000000   
              2017-03         91   2.334383     0.769525  0.000000   
              2017-04         89   4.798857     0.616956  0.000000   
              2017-05        105   5.936858     0.970764  0.000000   
              2017-06        145   5.998710     0.264884  0.000000   
              2017-07        144   6.323570     0.784444  0.000000   
              2017-08         67   4.683146     1.162604  0.000035   
              2017-09        160   6.875657     0.778096  0.000000   
              2017-10        153   3.386849     0.880903  0.000000   
              2017-11        109   3.688174     0.735949  0.000035   
              2017-12        105   3.056693     0.151227  0.000035   
              2018-01        238   8.112295     4.889468  0.000000   
              2018-02        297   7.999399     3.769931  0.000000   
              2018-03        381   6.958655     3.701065  0.000000   
              2018-04        459   6.355215     2.460243  0.000000   
              2018-05        397   8.420655     4.148727  0.000000   
              2018-06        572   8.845556     4.165035  0.000000   
              2018-07        543  10.541143     5.062106  0.000000   
              2018-08        335  11.734953     6.836238  0.000000   
              2018-09        535  10.683913     6.802454  0.000000   
              2018-10        522   7.538129     5.060463  0.000000   
              2018-11        472   8.068658     5.655943  0.000000   
              2018-12        357   6.894237     3.999942  0.000000   

                             max_days   p90_days  
log           start_month                         
domestic      2017-01      146.986713   7.124354  
              2017-02      337.101956   4.891584  
              2017-03       73.729653   4.023438  
              2017-04       24.163021   5.164847  
              2017-05      133.146019   7.004954  
              2017-06      362.876794   6.792120  
              2017-07      171.679826   6.756465  
              2017-08       37.719525   6.525898  
              20

Aufgabe 8

In [24]:
# Frage 8: Dateninventar bereits in Zelle 3 (field_summary) und Zelle 4 (Wertevielfalt).
# Essenz: Aktivitaet, Zeitstempel, Resource/Role sowie Fall-IDs und Finanz-/Permit-Attribute.
# Keine expliziten Ablehnungsgruende oder lifecycle:transition fuer Start/Ende der Arbeit.
# Daher keine saubere Trennung von Warte- und Bearbeitungszeit aus diesen Feldern.
# Welche weiteren Informationen fachlich benoetigt werden, ist keine reine Python-Berechnung.

Aufgabe 9

Aufgabe 10

Aufgabe 11